# 🏅 Strava Analytics — Desempenho Esportivo & Volume de Treino
**Foco**: Evolução de metas, recordes por modalidade e consistência de treino  
**Filtros ativos**: `DATE_START`, `DATE_END`, `SPORT_TYPES` — compatível com Streamlit

In [1]:
import sqlite3, warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
from IPython.display import display

warnings.filterwarnings('ignore')

# ── FILTROS PARAMETRIZÁVEIS (integração futura com Streamlit) ────────────────
DATE_START  = None     # ex: '2024-01-01'
DATE_END    = None     # ex: '2025-12-31'
SPORT_TYPES = None     # ex: ['Run', 'Ride'] ou None para todas

# Janelas de distância para análise de provas (metros)
RANGE_5K      = (4900,  5500)
RANGE_10K     = (9800, 10500)
RANGE_SWIM_1K = ( 900,  1100)

DB_PATH = Path('..') / 'data' / 'strava.db'
conn    = sqlite3.connect(DB_PATH)
print(f'✅ Conectado: {DB_PATH.resolve()}')

✅ Conectado: C:\Users\gabri\OneDrive\Belgeler\GitHub\strava-performance-analytics\data\strava.db


In [2]:
# ── CARGA CENTRAL ────────────────────────────────────────────────────────────
def load_activities(conn, date_start=None, date_end=None, sport_types=None):
    q, params = 'SELECT * FROM activities WHERE 1=1', []
    if date_start:
        q += ' AND start_date_local >= ?'; params.append(date_start)
    if date_end:
        q += ' AND start_date_local <= ?'; params.append(date_end + 'T23:59:59')
    if sport_types:
        ph = ','.join(['?'] * len(sport_types))
        q += f' AND sport_type IN ({ph})'; params.extend(sport_types)
    q += ' ORDER BY start_date_local'

    df = pd.read_sql(q, conn, params=params)
    df['start_date_local'] = pd.to_datetime(df['start_date_local'])
    df['month']       = df['start_date_local'].dt.to_period('M')
    df['month_str']   = df['month'].astype(str)
    df['year']        = df['start_date_local'].dt.year
    df['dist_km']     = df['distance'] / 1000
    df['duration_h']  = df['moving_time'] / 3600
    df['duration_min']= df['moving_time'] / 60
    df['speed_kmh']   = np.where(
        df['moving_time'] > 0, df['distance'] / df['moving_time'] * 3.6, np.nan)
    df['pace_min_km'] = np.where(
        (df['moving_time'] > 0) & (df['dist_km'] > 0),
        df['duration_min'] / df['dist_km'], np.nan)
    return df

def fmt_pace(p):
    """Converte pace decimal (min/km ou min/100m) para string MM:SS."""
    if pd.isna(p) or p <= 0: return 'N/A'
    m = int(p); s = int(round((p - m) * 60))
    if s == 60: m += 1; s = 0
    return f'{m}:{s:02d}'

df = load_activities(conn, DATE_START, DATE_END, SPORT_TYPES)
print(f'📊 {len(df)} atividades carregadas')
display(df['sport_type'].value_counts().rename('treinos').to_frame())

📊 353 atividades carregadas


,treinos
sport_type,
Walk,144
WeightTraining,72
Run,70
Swim,34
Ride,15
Soccer,10
Workout,4
Racquetball,1
Tennis,1


---
## 🥧 Pilar 1 — Volumetria Geral e Mix de Modalidades
> Mostra o **share** de cada esporte no histórico total. Revela se o atleta é generalista ou especializado e indica onde o volume está concentrado — base para balancear a distribuição de carga entre modalidades.

In [3]:
mix = (
    df.groupby('sport_type')
    .agg(treinos=('id','count'), dist_km=('dist_km','sum'), horas=('duration_h','sum'))
    .sort_values('treinos', ascending=False)
    .reset_index()
)
mix['pct']     = (mix['treinos'] / mix['treinos'].sum() * 100).round(1)
mix['dist_km'] = mix['dist_km'].round(1)
mix['horas']   = mix['horas'].round(1)

# ── Rosca: share de treinos ──
fig = px.pie(
    mix, names='sport_type', values='treinos', hole=0.55,
    title='Mix de Modalidades — Share de Treinos (%)',
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig.update_traces(textinfo='percent+label', pull=[0.04] * len(mix))
fig.update_layout(legend_title='Modalidade')
fig.show()

# ── Barras horizontais: volume em km (exclui esportes sem distância GPS) ──
mix_dist = mix[mix['dist_km'] > 0].sort_values('dist_km')
fig2 = px.bar(
    mix_dist, x='dist_km', y='sport_type', orientation='h',
    color='sport_type', text='dist_km',
    title='Volume Acumulado por Modalidade (km)',
    labels={'dist_km':'Distância Total (km)', 'sport_type':''},
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig2.update_traces(texttemplate='%{text:.0f} km', textposition='outside')
fig2.update_layout(showlegend=False, margin=dict(r=80))
fig2.show()

# ── Evolução mensal do mix (volume em horas) ──
mix_m = df.groupby(['month_str','sport_type'])['duration_h'].sum().reset_index()
fig3 = px.bar(
    mix_m, x='month_str', y='duration_h', color='sport_type', barmode='stack',
    title='Volume Mensal por Modalidade (horas)',
    labels={'month_str':'Mês','duration_h':'Horas','sport_type':'Modalidade'},
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig3.update_layout(xaxis_tickangle=-45)
fig3.show()

---
## 🏃 Pilar 2 — Recordes de Pace por Distância Alvo (5km e 10km)
> Isola sessões próximas de provas-padrão para medir performance real. O **PR** (Personal Record) é destacado com anotação. A diferença entre PR e pace médio revela a **margem de evolução** disponível.

In [4]:
RUN_TYPES = ['Run', 'TrailRun', 'VirtualRun']

def sessoes_prova(df, dist_range, label):
    low, high = dist_range
    d = df[
        df['sport_type'].isin(RUN_TYPES) &
        df['distance'].between(low, high) &
        df['pace_min_km'].between(2.5, 15)
    ].copy()
    d['label']    = label
    d['pace_str'] = d['pace_min_km'].apply(fmt_pace)
    return d

d5  = sessoes_prova(df, RANGE_5K,  '5 km')
d10 = sessoes_prova(df, RANGE_10K, '10 km')

for grupo, nome in [(d5, '5 km'), (d10, '10 km')]:
    if grupo.empty:
        print(f'⚠️  Nenhuma sessão de {nome} encontrada.'); continue
    pr      = grupo['pace_min_km'].min()
    media   = grupo['pace_min_km'].mean()
    pr_data = grupo.loc[grupo['pace_min_km'].idxmin(), 'start_date_local']
    margem  = (media - pr) / media * 100
    print(f'\n🏅 {nome} — {len(grupo)} sessões')
    print(f'   🥇 PR            : {fmt_pace(pr)}/km  ({pr_data.strftime("%d/%m/%Y")})')
    print(f'   📊 Pace médio    : {fmt_pace(media)}/km')
    print(f'   📈 Margem de melhora: {margem:.1f}% (diferença PR → médio)')

todos = pd.concat([d5, d10], ignore_index=True)
if not todos.empty:
    fig = px.scatter(
        todos, x='start_date_local', y='pace_min_km',
        color='label', symbol='label', size='dist_km',
        hover_data=['name','dist_km','pace_str'],
        title='Evolução de Pace — Sessões de 5km e 10km (destaque: PR)',
        labels={'start_date_local':'Data','pace_min_km':'Pace (min/km)','label':'Distância'},
        color_discrete_map={'5 km':'#26C6DA','10 km':'#7E57C2'}
    )
    fig.update_yaxes(autorange='reversed')
    for grupo, cor in [(d5,'#00838F'), (d10,'#4527A0')]:
        if grupo.empty: continue
        pr_row = grupo.loc[grupo['pace_min_km'].idxmin()]
        fig.add_annotation(
            x=pr_row['start_date_local'], y=pr_row['pace_min_km'],
            text=f"🥇 PR {pr_row['label']}: {pr_row['pace_str']}",
            showarrow=True, arrowhead=2, bgcolor=cor, font_color='white',
            borderpad=4, ax=40, ay=-30
        )
    fig.show()


🏅 5 km — 27 sessões
   🥇 PR            : 5:05/km  (26/07/2025)
   📊 Pace médio    : 6:29/km
   📈 Margem de melhora: 21.7% (diferença PR → médio)

🏅 10 km — 2 sessões
   🥇 PR            : 5:28/km  (22/06/2025)
   📊 Pace médio    : 5:59/km
   📈 Margem de melhora: 8.6% (diferença PR → médio)


---
## 📈 Pilar 3 — Curva de Evolução Histórica de Corrida 5K
> A linha de tendência **descendente** indica que o atleta está ficando mais rápido. A banda de variação mensal (pior → melhor pace) mostra consistência: banda estreita = treinos homogêneos; banda larga = alta variabilidade de esforço.

In [5]:
if d5.empty:
    print('⚠️  Nenhuma sessão de 5km encontrada no período.')
else:
    d5_m = (
        d5.groupby('month')
        .agg(pr_mensal=('pace_min_km','min'), media=('pace_min_km','mean'),
             pior=('pace_min_km','max'), n=('id','count'))
        .reset_index()
    )
    d5_m['month_str'] = d5_m['month'].astype(str)
    d5_m['pr_str']    = d5_m['pr_mensal'].apply(fmt_pace)
    d5_m['media_str'] = d5_m['media'].apply(fmt_pace)

    fig = go.Figure()

    # Banda de variação (melhor–pior pace mensal)
    x_band = list(d5_m['month_str']) + list(reversed(d5_m['month_str']))
    y_band = list(d5_m['pior'])      + list(reversed(d5_m['pr_mensal']))
    fig.add_trace(go.Scatter(
        x=x_band, y=y_band, fill='toself',
        fillcolor='rgba(38,198,218,0.12)',
        line=dict(color='rgba(0,0,0,0)'), name='Intervalo mensal'
    ))
    # Linha de pace médio mensal
    fig.add_trace(go.Scatter(
        x=d5_m['month_str'], y=d5_m['media'],
        mode='lines+markers', name='Pace médio mensal',
        line=dict(color='#26C6DA', width=2.5), marker=dict(size=7),
        customdata=d5_m[['media_str','n']],
        hovertemplate='%{x}<br>Médio: %{customdata[0]}<br>Sessões: %{customdata[1]}<extra></extra>'
    ))
    # Estrelas do PR mensal
    fig.add_trace(go.Scatter(
        x=d5_m['month_str'], y=d5_m['pr_mensal'],
        mode='markers', name='PR mensal',
        marker=dict(color='#FFA726', size=11, symbol='star'),
        customdata=d5_m[['pr_str']],
        hovertemplate='%{x}<br>PR: %{customdata[0]}<extra></extra>'
    ))

    fig.update_yaxes(autorange='reversed', title_text='Pace (min/km) — menor = mais rápido')
    fig.update_layout(
        title='Curva de Evolução Histórica — Corrida 5km',
        xaxis_title='Mês', xaxis_tickangle=-45, hovermode='x unified'
    )
    fig.show()

    if len(d5_m) >= 2:
        p_ini = d5_m['media'].iloc[0]
        p_fim = d5_m['media'].iloc[-1]
        delta = (p_ini - p_fim) / p_ini * 100
        sinal = '✅ melhora' if delta > 0 else '⚠️  piora'
        print(f'\n📊 Evolução: {fmt_pace(p_ini)} → {fmt_pace(p_fim)}  ({abs(delta):.1f}% de {sinal})')


📊 Evolução: 7:28 → 6:03  (19.0% de ✅ melhora)


---
## 🏊 Pilar 4 — Evolução da Natação (Endurance Aquático)
> O ritmo padrão de natação é **min/100m** (não min/km). Para atletas de triathlon, o parâmetro de referência é ≤ 2:00/100m para natação de travessia. A evolução do ritmo nas sessões de 1km indica adaptação aeróbica aquática.

In [6]:
df_swim = df[df['sport_type'].isin(['Swim', 'OpenWaterSwim'])].copy()

if df_swim.empty:
    print('⚠️  Nenhuma atividade de natação encontrada.')
else:
    df_swim['pace_100m'] = np.where(
        (df_swim['moving_time'] > 0) & (df_swim['distance'] > 0),
        (df_swim['moving_time'] / 60) / (df_swim['distance'] / 100),
        np.nan
    )
    df_swim = df_swim[df_swim['pace_100m'].between(0.8, 6)].copy()
    df_swim['pace_100m_str'] = df_swim['pace_100m'].apply(fmt_pace)

    print(f'🏊 {len(df_swim)} sessões | Volume total: {df_swim["dist_km"].sum():.1f} km')
    print(f'   PR: {fmt_pace(df_swim["pace_100m"].min())}/100m | '
          f'Médio: {fmt_pace(df_swim["pace_100m"].mean())}/100m')

    # Volume e ritmo mensal
    sw_m = df_swim.groupby('month_str').agg(
        dist_km=('dist_km','sum'), sessoes=('id','count'), pace_medio=('pace_100m','mean')
    ).reset_index()
    sw_m['pace_str'] = sw_m['pace_medio'].apply(fmt_pace)

    fig = make_subplots(specs=[[{'secondary_y': True}]])
    fig.add_trace(go.Bar(
        x=sw_m['month_str'], y=sw_m['dist_km'],
        name='Volume (km)', marker_color='#4FC3F7', opacity=0.7
    ), secondary_y=False)
    fig.add_trace(go.Scatter(
        x=sw_m['month_str'], y=sw_m['pace_medio'],
        name='Pace médio (min/100m)', mode='lines+markers',
        line=dict(color='#E91E63', width=2.5),
        customdata=sw_m[['pace_str']],
        hovertemplate='%{x}<br>Pace: %{customdata[0]}/100m<extra></extra>'
    ), secondary_y=True)
    fig.update_yaxes(title_text='Volume (km)', secondary_y=False)
    fig.update_yaxes(title_text='Pace (min/100m) ↓ melhor', autorange='reversed', secondary_y=True)
    fig.update_layout(title='Natação — Volume Mensal e Evolução de Ritmo', xaxis_tickangle=-45)
    fig.show()

    # Sessões de 1km isoladas
    sw_1k = df_swim[df_swim['distance'].between(*RANGE_SWIM_1K)].copy()
    if not sw_1k.empty:
        print(f'\n🎯 Sessões próximas de 1km: {len(sw_1k)}')
        print(f'   PR: {fmt_pace(sw_1k["pace_100m"].min())}/100m | '
              f'Médio: {fmt_pace(sw_1k["pace_100m"].mean())}/100m')
        fig2 = px.scatter(
            sw_1k, x='start_date_local', y='pace_100m',
            hover_data=['name','pace_100m_str','dist_km'],
            title='Evolução do Ritmo em Sessões de ~1km (Natação)',
            labels={'start_date_local':'Data','pace_100m':'Pace (min/100m)'},
            trendline='lowess', color_discrete_sequence=['#E91E63']
        )
        fig2.update_yaxes(autorange='reversed')
        fig2.show()
    else:
        print('⚠️  Nenhuma sessão próxima de 1km. Ajuste RANGE_SWIM_1K se necessário.')

🏊 33 sessões | Volume total: 36.5 km
   PR: 1:17/100m | Médio: 2:33/100m



🎯 Sessões próximas de 1km: 12
   PR: 1:54/100m | Médio: 2:15/100m


---
## 🏋️ Pilar 5 — Consistência de Força (Academia / WeightTraining)
> Na musculação não há GPS. A métrica é o **tempo acumulado mensal**. A linha de meta (6h/mês ≈ 3 sessões de 2h) serve como referência de consistência mínima de treino de força para atletas de endurance — abaixo disso há risco de perda muscular.

In [7]:
FORCE_TYPES  = ['WeightTraining', 'Workout', 'Crossfit', 'Yoga', 'Pilates', 'RockClimbing']
META_HORAS   = 6    # meta mensal mínima de horas de força

df_force = df[df['sport_type'].isin(FORCE_TYPES)].copy()

if df_force.empty:
    print('⚠️  Nenhuma atividade de força encontrada.')
else:
    print(f'💪 {len(df_force)} sessões | {df_force["sport_type"].value_counts().to_dict()}')
    print(f'   Total acumulado: {df_force["duration_h"].sum():.1f}h')

    force_m = (
        df_force.groupby(['month_str', 'sport_type'])
        .agg(horas=('duration_h','sum'), sessoes=('id','count'))
        .reset_index()
    )

    # Barras empilhadas: horas mensais por tipo de força
    fig = px.bar(
        force_m, x='month_str', y='horas', color='sport_type', barmode='stack',
        title='Consistência de Força — Horas Mensais Acumuladas',
        labels={'month_str':'Mês','horas':'Horas','sport_type':'Modalidade'},
        color_discrete_sequence=px.colors.qualitative.Pastel,
        text_auto='.1f'
    )
    fig.add_hline(
        y=META_HORAS, line_dash='dash', line_color='crimson',
        annotation_text=f'Meta mínima: {META_HORAS}h/mês',
        annotation_position='top right'
    )
    fig.update_layout(xaxis_tickangle=-45)
    fig.show()

    # Resumo de consistência
    mensal = df_force.groupby('month_str')['duration_h'].sum()
    pct_ok = (mensal >= META_HORAS).mean() * 100
    print(f'\n📊 {pct_ok:.0f}% dos meses atingiram ≥ {META_HORAS}h de força')
    print(f'   Média mensal: {mensal.mean():.1f}h | Máximo: {mensal.max():.1f}h | Mínimo: {mensal.min():.1f}h')

    # Tendência: média móvel de 3 meses
    mensal_df = mensal.reset_index()
    mensal_df.columns = ['month_str', 'horas']
    mensal_df['media_3m'] = mensal_df['horas'].rolling(3, min_periods=1).mean()

    fig2 = go.Figure()
    fig2.add_trace(go.Bar(
        x=mensal_df['month_str'], y=mensal_df['horas'],
        name='Horas/mês', marker_color='#CE93D8', opacity=0.7
    ))
    fig2.add_trace(go.Scatter(
        x=mensal_df['month_str'], y=mensal_df['media_3m'],
        name='Média móvel 3 meses', mode='lines',
        line=dict(color='#7B1FA2', width=2.5, dash='dot')
    ))
    fig2.add_hline(y=META_HORAS, line_dash='dash', line_color='crimson',
                   annotation_text=f'Meta: {META_HORAS}h')
    fig2.update_layout(
        title='Tendência de Consistência de Força (Média Móvel 3 Meses)',
        xaxis_title='Mês', yaxis_title='Horas', xaxis_tickangle=-45
    )
    fig2.show()

💪 77 sessões | {'WeightTraining': 72, 'Workout': 4, 'RockClimbing': 1}
   Total acumulado: 64.8h



📊 29% dos meses atingiram ≥ 6h de força
   Média mensal: 4.6h | Máximo: 14.6h | Mínimo: 0.5h


---
## 🚴 Pilar 6 — Ciclismo: Eficiência em Subidas (Elevação × Velocidade)
> O **quadrante superior direito** do scatter (alta elevação + alta velocidade) identifica os treinos de subida mais eficientes — aqueles onde o atleta sustentou boa velocidade apesar do desnível positivo. São os treinos de maior valor de estímulo para ciclismo de montanha ou triathlon.

In [8]:
RIDE_TYPES = ['Ride', 'VirtualRide', 'MountainBikeRide', 'GravelRide', 'EBikeRide']
df_ride = df[
    df['sport_type'].isin(RIDE_TYPES) &
    (df['total_elevation_gain'] > 0) &
    df['speed_kmh'].between(5, 65)
].copy()

if df_ride.empty:
    print('⚠️  Nenhuma atividade de ciclismo com elevação encontrada.')
else:
    df_ride['elev_per_km'] = (df_ride['total_elevation_gain'] / df_ride['dist_km']).round(1)
    df_ride['data_str']    = df_ride['start_date_local'].dt.strftime('%d/%m/%Y')

    print(f'🚴 {len(df_ride)} atividades de ciclismo com dados de elevação')

    med_elev  = df_ride['total_elevation_gain'].median()
    med_speed = df_ride['speed_kmh'].median()

    # Scatter principal com quadrantes
    fig = px.scatter(
        df_ride,
        x='total_elevation_gain', y='speed_kmh',
        size='dist_km', color='year',
        hover_data=['name','data_str','dist_km','elev_per_km'],
        trendline='lowess',
        title='Ciclismo — Ganho de Elevação vs Velocidade Média  (bolhas = distância)',
        labels={'total_elevation_gain':'Ganho de Elevação (m)',
                'speed_kmh':'Velocidade Média (km/h)', 'year':'Ano'},
        color_continuous_scale='Viridis'
    )
    fig.add_vline(x=med_elev,  line_dash='dot', line_color='gray', opacity=0.6,
                  annotation_text=f'Mediana elevação ({med_elev:.0f}m)',
                  annotation_position='top right')
    fig.add_hline(y=med_speed, line_dash='dot', line_color='gray', opacity=0.6,
                  annotation_text=f'Mediana velocidade ({med_speed:.1f} km/h)')
    fig.show()

    # Top 10 subidas mais eficientes (velocidade >= mediana)
    top_climb = (
        df_ride[df_ride['speed_kmh'] >= med_speed]
        .nlargest(10, 'elev_per_km')
        [['data_str','name','dist_km','total_elevation_gain','elev_per_km','speed_kmh']]
        .copy()
    )
    top_climb.columns = ['Data','Atividade','Dist(km)','Elevação(m)','Elev/km','Vel(km/h)']
    top_climb = top_climb.round(1)
    print('\n🏆 Top 10 treinos de subida mais eficientes (velocidade ≥ mediana):')
    display(top_climb.to_string(index=False))

    # Evolução mensal: volume e velocidade média
    ride_m = df_ride.groupby('month_str').agg(
        dist_total=('dist_km','sum'), speed_medio=('speed_kmh','mean'),
        elev_total=('total_elevation_gain','sum'), n=('id','count')
    ).reset_index()

    fig2 = make_subplots(specs=[[{'secondary_y': True}]])
    fig2.add_trace(go.Bar(
        x=ride_m['month_str'], y=ride_m['dist_total'],
        name='Volume (km)', marker_color='#81C784', opacity=0.7
    ), secondary_y=False)
    fig2.add_trace(go.Scatter(
        x=ride_m['month_str'], y=ride_m['speed_medio'],
        name='Velocidade Média (km/h)', mode='lines+markers',
        line=dict(color='#FF8F00', width=2.5)
    ), secondary_y=True)
    fig2.update_yaxes(title_text='Volume (km)', secondary_y=False)
    fig2.update_yaxes(title_text='Velocidade Média (km/h)', secondary_y=True)
    fig2.update_layout(
        title='Ciclismo — Volume Mensal e Evolução de Velocidade',
        xaxis_tickangle=-45
    )
    fig2.show()

🚴 14 atividades de ciclismo com dados de elevação



🏆 Top 10 treinos de subida mais eficientes (velocidade ≥ mediana):


'      Data    Atividade  Dist(km)  Elevação(m)  Elev/km  Vel(km/h)\n09/11/2025 Morning Ride      10.9        136.0     12.5       15.0\n05/10/2025 Morning Ride      22.4        272.0     12.1       17.1\n19/07/2025   Lunch Ride      10.8        127.0     11.7       15.7\n13/05/2026 Morning Ride      29.5        268.0      9.1       13.1\n27/07/2025 Morning Ride      20.1        169.0      8.4       11.3\n12/10/2025 Morning Ride      22.1        185.0      8.4       17.5\n03/08/2025 Morning Ride      24.8        196.0      7.9       16.9'